In [1]:
from src.utils import *

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm import tqdm

import torch

import scipy
import librosa

from sklearn.model_selection import train_test_split

# from pyannote.audio import Pipeline
# from pyannote.audio.pipelines.utils.hook import ProgressHook

/home/zakhar/miniconda3/envs/aphasia_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1786906079.036084 1437832 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786906079.074094 1437832 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786906080.079226 1437832 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to 

In [2]:
DATA_DIR = os.path.join(os.getcwd(), 'data')
VOICES_DIR = os.path.join(DATA_DIR, 'Voices_wav')
APHASIA_DIR = os.path.join(VOICES_DIR, 'Aphasia')
NORM_DIR = os.path.join(VOICES_DIR, 'Norm')

In [3]:
meta_data_aphasia = pd.read_excel(os.path.join(DATA_DIR, "DemographicData.xlsx"), sheet_name='пациенты_revised')
meta_data_norm = pd.read_excel(os.path.join(DATA_DIR, "DemographicData.xlsx"), sheet_name='норма')

meta_data_norm.drop(meta_data_norm.index[-1], axis=0, inplace=True)
meta_data_aphasia["Subj_ID"] = meta_data_aphasia["Subj_ID"].astype(str)
meta_data_norm.rename(columns={"№": "Subj_ID"}, inplace=True)
meta_data_norm["Subj_ID"] = meta_data_norm["Subj_ID"].astype(str)

In [4]:
meta_data_aphasia_1 = pd.read_excel(os.path.join(DATA_DIR, "Demo RAT discourse production.xlsx"), sheet_name='пациенты')
meta_data_aphasia_1["Subj_ID"] = meta_data_aphasia_1["Subj_ID"].astype(str)

In [5]:
meta_data_aphasia.rename(columns={'Stroked hemisphere (L/R/LR)': 'Stroked hemisphere',
                          'Aphasia/Norm (A/N), A includes all patients, even with just dysarthria': 'Aphasia/Norm', 
                          'Aphasia_Severity_Three_classes \n0 - mild\n1 - moderate\n2 -severe\nNA - no aphasia': 'Aphasia_Severity',
                          'Aphasia_Types\n1 - efferent motor\n2 - afferent motor\n3 - complex motor\n4 - dynamic\n5 - acoustic-mnestic\n6 - sensory\n7 - semantic\n8 - other': 'Aphasia_Types',
                          'Dominant aphasia type: detailed\n0 - motor\n1- dynamic\n2 - sensory\n3 - acoustic-mnestic\nNA - no aphasia or small classes (semantic, amnestic, conduction)': 'Dominant_Aphasia',
                          'Dysarthria (1/0)': 'Dysarthria',
                          'Dysarthria_Severity_Three_classes \n0 - mild\n1 - moderate\n2 -severe\nNA - no dysarthria': 'Dysarthria_Severity',
                          }, inplace=True)

In [6]:
meta_data_aphasia_1.rename(columns={'Stroked hemisphere (L/R/LR)': 'Stroked hemisphere',
                          'Aphasia/Norm (A/N), A includes all patients, even with just dysarthria': 'Aphasia/Norm', 
                          'Aphasia_Severity \n0 - no aphasia\n1 - very mild\n2 - mild\n3 - mild-moderate\n4 - moderate\n5 - moderate-severe\n6 - severe\n7 - very severe': 'Aphasia_Severity',
                          'Aphasia_Types\n1 - efferent motor\n2 - afferent motor\n3 - complex motor\n4 - dynamic\n5 - acoustic-mnestic\n6 - sensory\n7 - semantic\n8 - other': 'Aphasia_Types',
                          'Dominant_Aphasia\n1 - efferent motor\n2 - afferent motor\n3 - complex motor\n4 - dynamic\n5 - acoustic-mnestic\n6 - sensory\n7 - semantic\n8 - other': 'Dominant_Aphasia',
                          'Dysarthria (1/0)': 'Dysarthria',
                          'Dysarthria_Severity \n0 - no dysarthria\n1 - very mild\n2 - mild\n3 - mild-moderate\n4 - moderate\n5 - moderate-severe\n6 - severe\n7 - very severe': 'Dysarthria_Severity',
                          }, inplace=True)

In [7]:
result = meta_data_aphasia_1[~meta_data_aphasia_1["Subj_ID"].isin(meta_data_aphasia["Subj_ID"])]

In [8]:
meta_data_aphasia = pd.concat([meta_data_aphasia, result], axis=0, ignore_index=True)

In [9]:
sf_aphasia = []
sf_norm = []

noise_aphasia = {}
noise_norm = {}

empty_files = []

In [10]:
for audio_file in tqdm(os.listdir(APHASIA_DIR)):
    y, sr = librosa.load(os.path.join(APHASIA_DIR, audio_file), sr=None)
    if len(y) == 0:
        print(audio_file)
    duration = librosa.get_duration(y=y, sr=sr)
    
    spectral_flatness = np.mean(librosa.feature.spectral_flatness(y=y))
    roll_off = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    
    if spectral_flatness > 0.5:
        empty_files.append(audio_file)
        continue


    ind = audio_file.split('-')[1]

    sf_aphasia.append([audio_file, spectral_flatness, roll_off])
    if ind not in noise_aphasia:
        noise_aphasia[str(ind)] = False
    if roll_off < 1_000:
        noise_aphasia[str(ind)] = True

100%|██████████| 608/608 [00:35<00:00, 17.15it/s]


In [11]:
for audio_file in tqdm(os.listdir(NORM_DIR)):
    y, sr = librosa.load(os.path.join(NORM_DIR, audio_file), sr=None)
    duration = librosa.get_duration(y=y, sr=sr)
    
    spectral_flatness = np.mean(librosa.feature.spectral_flatness(y=y))
    roll_off = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    
    if spectral_flatness > 0.5:
        empty_files.append(audio_file)
        continue
    
    sf_norm.append([audio_file, spectral_flatness, roll_off])
    ind = audio_file.split('-')[1]
    if ind[0] == '0':
        ind = ind[1:]
    ind += '-НРАТ'
    
    if ind not in noise_norm:
        noise_norm[str(ind)] = False
    if roll_off < 1_000:
        noise_norm[str(ind)] = True

100%|██████████| 202/202 [00:04<00:00, 49.12it/s]


In [12]:
[x for x in noise_aphasia.keys() if x not in meta_data_aphasia["Subj_ID"].tolist()]

['409', '426', 'L45']

In [13]:
[x for x in noise_norm.keys() if x not in meta_data_norm["Subj_ID"].tolist()]

[]

In [14]:
meta_data_norm = meta_data_norm[meta_data_norm["Subj_ID"].isin(noise_norm.keys())]
meta_data_norm.shape

(101, 17)

In [15]:
meta_data_aphasia.shape

(378, 43)

In [16]:
meta_data_aphasia = meta_data_aphasia[meta_data_aphasia["Subj_ID"].isin(noise_aphasia.keys())]
meta_data_aphasia.shape

(253, 43)

In [17]:
meta_data_aphasia["has_noise"] = meta_data_aphasia["Subj_ID"].map(noise_aphasia)
meta_data_norm["has_noise"] = meta_data_norm["Subj_ID"].map(noise_norm)

In [18]:
no_aphasia_ids = meta_data_aphasia[(meta_data_aphasia["Aphasia_Severity"].isna()) & (meta_data_aphasia["Dysarthria_Severity"].isna())]["Subj_ID"].values
no_aphasia_ids

array(['1359', '251', '1130', '203', '754', '878', '1218', '1231', '1237',
       '1462', '1467', '1468'], dtype=object)

In [19]:
meta_data_aphasia[meta_data_aphasia["Subj_ID"].isin(no_aphasia_ids)][["Subj_ID", "Aphasia_Severity"]]

,Subj_ID,Aphasia_Severity
360,1359,NaN
361,251,NaN
362,1130,NaN
363,203,NaN
367,754,NaN
368,878,NaN
371,1218,NaN
372,1231,NaN
373,1237,NaN
375,1462,NaN


In [20]:
meta_data_norm.shape, meta_data_aphasia.shape

((101, 18), (253, 44))

In [21]:
meta_data_aphasia[meta_data_aphasia["Subj_ID"].isin(no_aphasia_ids)].shape

(12, 44)

In [22]:
meta_data_aphasia[meta_data_aphasia["Subj_ID"].isin(no_aphasia_ids)].index.shape

(12,)

In [23]:
meta_data_aphasia.drop(meta_data_aphasia[meta_data_aphasia["Subj_ID"].isin(no_aphasia_ids)].index, axis=0, inplace=False).shape

(241, 44)

In [24]:
meta_data_norm = pd.concat([meta_data_norm, meta_data_aphasia[meta_data_aphasia["Subj_ID"].isin(no_aphasia_ids)]], axis=0)
meta_data_aphasia.drop(meta_data_aphasia[meta_data_aphasia["Subj_ID"].isin(no_aphasia_ids)].index, axis=0, inplace=True)

In [25]:
meta_data_norm.shape, meta_data_aphasia.shape

((113, 60), (241, 44))

In [26]:
train_aphasia, test_aphasia = train_test_split(meta_data_aphasia, test_size=0.4, random_state=42, stratify=meta_data_aphasia["has_noise"])

val_aphasia, test_aphasia = train_test_split(test_aphasia, test_size=0.5, random_state=42, stratify=test_aphasia["has_noise"])

In [27]:
train_norm, test_norm = train_test_split(meta_data_norm, test_size=0.4, random_state=42, stratify=meta_data_norm["has_noise"])

val_norm, test_norm = train_test_split(test_norm, test_size=0.5, random_state=42, stratify=test_norm["has_noise"])

In [28]:
train_aphasia.shape, val_aphasia.shape, test_aphasia.shape

((144, 44), (48, 44), (49, 44))

In [29]:
train_norm.shape, val_norm.shape, test_norm.shape

((67, 60), (23, 60), (23, 60))

In [30]:
aphasia_files_list = os.listdir(APHASIA_DIR)
norm_files_list = os.listdir(NORM_DIR)

In [31]:
aphasia_files_dict = {}
norm_files_dict = {}

for audio_file in tqdm(aphasia_files_list):
    participant_id = audio_file.split('-')[1]
    if participant_id in no_aphasia_ids:
        if participant_id not in norm_files_dict:
            norm_files_dict[participant_id] = [audio_file]
        else:
            norm_files_dict[participant_id].append(audio_file)
    else:
        if participant_id not in aphasia_files_dict:
            aphasia_files_dict[participant_id] = [audio_file]
        else:
            aphasia_files_dict[participant_id].append(audio_file)

for audio_file in tqdm(norm_files_list):
    participant_id = audio_file.split('-')[1]
    if participant_id not in norm_files_dict:
        norm_files_dict[participant_id] = [audio_file]
    else:
        norm_files_dict[participant_id].append(audio_file)

100%|██████████| 202/202 [00:00<00:00, 1681050.41it/s]


In [32]:
norm_files_dict

{'1467': ['A-1467-RAT-1-bike.wav',
  'A-1467-RAT-2-bike.wav',
  'A-1467-RAT-1-robb.wav',
  'A-1467-RAT-2-robb.wav'],
 '1468': ['A-1468-RAT-2-bike.wav',
  'A-1468-RAT-2-robb.wav',
  'A-1468-RAT-1-bike.wav',
  'A-1468-RAT-1-robb.wav'],
 '1237': ['A-1237-RAT-1-bike.wav', 'A-1237-RAT-1-robb.wav'],
 '1462': ['A-1462-RAT-1-robb.wav',
  'A-1462-RAT-1-bike.wav',
  'A-1462-RAT-2-robb.wav',
  'A-1462-RAT-2-bike.wav',
  'A-1462-RAT-3-robb.wav',
  'A-1462-RAT-1-robb(1).wav',
  'A-1462-RAT-3-bike.wav'],
 '1231': ['A-1231-RAT-1-robb.wav', 'A-1231-RAT-1-bike.wav'],
 '1359': ['A-1359-RAT-1-bike.wav'],
 '1130': ['A-1130-RAT-1-bike.wav'],
 '1218': ['A-1218-RAT-1-robb.wav', 'A-1218-RAT-1-bike.wav'],
 '203': ['A-203-RAT-1-bike.wav',
  'A-203-RAT-2-robb.wav',
  'A-203-RAT-2-bike.wav',
  'A-203-RAT-1-robb.wav'],
 '754': ['A-754-RAT-1-bike.wav', 'A-754-RAT-2-bike.wav'],
 '251': ['A-251-RAT-1-bike.wav', 'A-251-RAT-1-robb.wav'],
 '878': ['A-878-RAT-1-bike.wav'],
 '0941': ['N-0941-RAT-1-bike.wav', 'N-0941-RAT-1

In [33]:
def get_filename(df: pd.DataFrame, label: int):
    output = []
    file_columns = [x for x in df.columns if "file_name" in x]
    for i, row in df.iterrows():
        participant_id = row['Subj_ID'].split('-')[0]
        # print(participant_id)
        if label:
            file_names = [(x, label) for x in  aphasia_files_dict[participant_id] if x not in empty_files]
        else:
            # print(row)
            if len(participant_id) == 3 and participant_id not in no_aphasia_ids:
                participant_id = '0' + participant_id
                
            file_names = [(x, label) for x in  norm_files_dict[participant_id] if x not in empty_files]
        output.extend(file_names)
    return output

In [34]:
train_filenames = get_filename(train_norm, 0) + get_filename(train_aphasia, 1)
val_filenames = get_filename(val_norm, 0) + get_filename(val_aphasia, 1)
test_filenames = get_filename(test_norm, 0) + get_filename(test_aphasia, 1)

In [35]:
len(train_filenames), len(val_filenames), len(test_filenames)

(478, 157, 168)

In [36]:
train_filenames = pd.DataFrame(train_filenames, columns=['file_name', 'label'])
val_filenames = pd.DataFrame(val_filenames, columns=['file_name', 'label'])
test_filenames = pd.DataFrame(test_filenames, columns=['file_name', 'label'])

In [37]:
full_df = pd.concat([train_filenames, val_filenames, test_filenames], axis=0, ignore_index=True)
full_df["file_name"].duplicated().sum()

np.int64(0)

In [38]:
train_filenames.to_csv(os.path.join(DATA_DIR, "train_filenames.csv"), index=False)
val_filenames.to_csv(os.path.join(DATA_DIR, "val_filenames.csv"), index=False)
test_filenames.to_csv(os.path.join(DATA_DIR, "test_filenames.csv"), index=False)

In [39]:
train_aphasia.to_csv(os.path.join(DATA_DIR, "train_aphasia.csv"), index=False)
val_aphasia.to_csv(os.path.join(DATA_DIR, "val_aphasia.csv"), index=False)
test_aphasia.to_csv(os.path.join(DATA_DIR, "test_aphasia.csv"), index=False)

train_norm.to_csv(os.path.join(DATA_DIR, "train_norm.csv"), index=False)
val_norm.to_csv(os.path.join(DATA_DIR, "val_norm.csv"), index=False)
test_norm.to_csv(os.path.join(DATA_DIR, "test_norm.csv"), index=False)

In [40]:
empty_files

['A-928-RAT-1-robb.wav']